<a href="https://colab.research.google.com/github/renan-figueira/RS-w-Funk-SVD-and-Cosine-correlation/blob/main/main" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Why use numpy < 2.0?**

The surprise library, which contains the Funk SVD model, has not yet been updated for the most recent versions of numpy, which is why an older version of the library is used.

In [2]:
!pip install "numpy<2.0" scikit-surprise

In [4]:
import pandas as pd
import numpy as np
from google.colab import drive
from surprise import Dataset, Reader, SVD
from sklearn.metrics.pairwise import cosine_similarity

# 1. SETUP AND DATA LOADING

In [5]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [6]:
# Path to the dataset folder
path = '/content/drive/MyDrive/sistema de recomendacao/archive/ml-100k/'

In [7]:
# Loading the ratings
rating_columns = ['user_id', 'item_id', 'rating', 'timestamp']
df_ratings = pd.read_csv(f'{path}u.data', sep='\t', names=rating_columns)

In [8]:
# Loading the movies (FIXED: added encoding='latin-1')
movie_columns = ['item_id', 'title']
df_movies = pd.read_csv(
    f'{path}u.item',
    sep='|',
    names=movie_columns,
    usecols=[0, 1],
    encoding='latin-1'
)

In [9]:
# Mapping dictionaries (Id -> Title and Title -> Id)
id_to_title = pd.Series(df_movies.title.values, index=df_movies.item_id).to_dict()
title_to_id = pd.Series(df_movies.item_id.values, index=df_movies.title).to_dict()

# 2. SVD MODEL TRAINING

In [10]:
print("\n[1/3] Preparing data for Surprise...")
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df_ratings[['user_id', 'item_id', 'rating']], reader)
full_trainset = data.build_full_trainset()


[1/3] Preparing data for Surprise...


In [11]:
print("[2/3] Training the SVD model (Latent Factors)...")
algo = SVD(n_factors=100, n_epochs=20, random_state=42)
algo.fit(full_trainset)
print("Training completed!")

[2/3] Training the SVD model (Latent Factors)...
Training completed!


# 3. RECOMMENDATION FUNCTION (Latent Factors + Cosine)

In [12]:
def recommend_similar_movies_svd(movie_title, top_n=10):
    print(f"\n[3/3] Finding the {top_n} most similar movies to: '{movie_title}'\n")

    # Check if the input movie exists in the database
    if movie_title not in title_to_id:
        return f"Error: Movie '{movie_title}' not found. Please check the spelling."

    # Get the Original (Raw) ID and convert it to the model's Inner ID
    movie_raw_id = title_to_id[movie_title]
    movie_inner_id = full_trainset.to_inner_iid(movie_raw_id)

    # 1. Matrix with the latent factors of all movies ('algo.qi')
    item_factors = algo.qi

    # 2. Specific factors of our target movie
    movie_vector = item_factors[movie_inner_id]

    # 3. Calculate cosine similarity (target movie vs all movies)
    similarities = cosine_similarity([movie_vector], item_factors)[0]

    # 4. Associate similarities with IDs, sort, and slice the top N
    sim_scores = list(enumerate(similarities))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # We take from 1 to top_n+1 to skip index 0 (which is the target movie itself with a score of 1.0)
    top_sim_scores = sim_scores[1:top_n+1]

    # 5. Map the results and display
    for inner_id, score in top_sim_scores:
        raw_id = int(full_trainset.to_raw_iid(inner_id))
        title = id_to_title.get(raw_id, f"ID {raw_id} Unknown")
        print(f"{title} (Similarity: {score:.4f})")

# 4. EXECUTION

In [17]:
recommend_similar_movies_svd('Critical Care (1997)', top_n=10)

# Try uncommenting the line below to quickly test another movie:
# recommend_similar_movies_svd('Toy Story (1995)', top_n=5)


[3/3] Finding the 10 most similar movies to: 'Critical Care (1997)'

Umbrellas of Cherbourg, The (Parapluies de Cherbourg, Les) (1964) (Similarity: 0.3275)
Fall (1997) (Similarity: 0.3160)
Schizopolis (1996) (Similarity: 0.2904)
Lawnmower Man 2: Beyond Cyberspace (1996) (Similarity: 0.2802)
Secret Adventures of Tom Thumb, The (1993) (Similarity: 0.2716)
Year of the Horse (1997) (Similarity: 0.2666)
Last Time I Committed Suicide, The (1997) (Similarity: 0.2622)
Clean Slate (1994) (Similarity: 0.2602)
Kiss the Girls (1997) (Similarity: 0.2539)
Collectionneuse, La (1967) (Similarity: 0.2533)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')